In [ ]:
!pip install -U llama-cpp-python

In [ ]:
import huggingface_hub as hf
import dotenv
import os

dotenv.load_dotenv()
hf.login(token=os.getenv("HF_TOKEN"))

In [ ]:
import gc, torch
for n in ["model", "inputs", "outputs", "processor"]:
    globals().pop(n, None)
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"free: {free/1e9:.2f} / {total/1e9:.2f} GiB")

In [ ]:
# import torch
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

# MODEL_ID = "unsloth/Qwen2.5-VL-3B-Instruct-unsloth-bnb-4bit"

# # Using the model's default pixel range. To cap resolution later (e.g. if you OOM),
# # set it on the processor without reloading the model:
# #   processor.image_processor.max_pixels = 1024 * 28 * 28
# processor = AutoProcessor.from_pretrained(MODEL_ID)

# # IMPORTANT: needs transformers<5 (tested on 4.57). transformers 5.x breaks the
# # 4-bit bitsandbytes loading for this checkpoint (all_tied_weights_keys / bnb assert).
# # The class is Qwen2_5_VLForConditionalGeneration (there is no AutoModelForMultimodalLM).
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     device_map="auto",
#     dtype="auto",
# ).eval()

# free, total = torch.cuda.mem_get_info()
# print(f"loaded. gpu free: {free/1e9:.2f} / {total/1e9:.2f} GiB")

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
MODEL_ID = "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit"
# default: Load the model on the available device(s)
processor = AutoProcessor.from_pretrained(MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen3VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen3-VL-4B-Instruct",
#     dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

In [ ]:
import torch
free, total = torch.cuda.mem_get_info()
print(f"loaded. gpu free: {free/1e9:.2f} / {total/1e9:.2f} GiB")

In [ ]:
from PIL import Image


def caption(image_path, user_prompt, system_prompt=None, max_new_tokens=320):
    """Run the VLM on one image and return the raw text response.

    Re-run this + the test cell freely WITHOUT reloading the model.
    Pass system_prompt to steer behaviour (grounding, color binding, output format).
    """
    image = Image.open(image_path).convert("RGB")
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": user_prompt},
    ]})
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(
        out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

In [ ]:
PROMPT = """
You are an expert fashion vision-language model that extracts structured metadata for a multimodal fashion retrieval system.

Your goal is to produce consistent, retrieval-friendly metadata describing the PRIMARY person in the image.

GENERAL RULES

- Return ONLY valid JSON matching the schema exactly, with no markdown or extra text.
- Base every attribute on visible evidence only.
- Describe a garment when you can identify it with reasonable confidence, even if partially occluded.
- Use null for any attribute you cannot determine, and [] for any empty list.
- Use concise, consistent terminology across images.

SCHEMA COMPLETENESS

Always include EVERY key from the schema, in the exact order shown, in every response.
Fill unknown scalar fields with null and unknown list fields with []. Keep the top/bottom keys present even when the outfit is a one-piece (set them to null / []).

INSPECTION ORDER

Inspect the primary person from head to toe: head/hair accessories, neck, upper body, outerwear, lower body, footwear, accessories (bags, watches, jewelry, belts, hats, glasses), pose/action, then scene/background.

ONE-PIECE RULE

For a one-piece outfit (dress, gown, jumpsuit, romper, bodysuit worn as the primary garment):
- Fill dress, dress_color, dress_features.
- Set top, top_color, bottom, bottom_color to null and top_features, bottom_features to [].

For separates, fill top and bottom, and set dress = null, dress_color = null, dress_features = [].

GARMENT RULES

The garment fields are: top, bottom, dress, outerwear, shoes.

- Write each garment as a garment NOUN, optionally with a fit or style word
  (e.g. "oversized hoodie", "cropped denim jacket", "sleeveless tank top", "wide-leg trousers",
   "straight-leg jeans", "pleated skirt", "bomber jacket", "combat boots", "strapless cocktail dress").
- Put the garment's color in its matching *_color field, and keep the garment field itself color-free.
    write:  "dress": "cocktail dress", "dress_color": "black"   (not "dress": "black dress")
- Put patterns and fits in *_features, and always keep a noun in the garment field.
    write:  "dress": "polka dot dress", "dress_features": ["polka dot"]   (not "dress": "polka dot")
- When you can only see a sleeve, fit, or pattern, combine it with the best-guess noun
  (e.g. "long-sleeved top", "sleeveless dress"). If the type is unclear, use a generic noun:
  top, bottom, dress, jacket, or shoes.

COLORS

Store each color only in its *_color field, as a plain dominant color name
(e.g. black, white, navy, olive, beige). Choose the closest concrete color; use null if truly uncertain.

FEATURES

Use *_features for visual characteristics such as: striped, floral, plaid, polka dot, ruched,
pleated, lace, embroidered, ruffled, cropped, oversized, fitted, distressed, sheer, quilted, ribbed.
Keep colors and the garment noun out of the features list.

ACCESSORIES

List every clearly visible accessory as an object with "item" and "color". Use [] when there are none.

  [{"item": "watch", "color": "gold"}, {"item": "crossbody bag", "color": "brown"}]

STYLE

Give the overall fashion style as a short phrase
(e.g. casual streetwear, business formal, elegant evening, minimalist casual, athleisure, bohemian, smart casual).

ENVIRONMENT

Give the scene as ONE short canonical phrase, using the shortest form
(e.g. city street, park, fashion runway, studio, beach, shopping mall, office, living room, red carpet).

ACTION

Give the person's primary action as a single verb, preferring:
standing, walking, posing, sitting, running, jumping, talking.
Use a short verb phrase for anything else, or null when no person/action is visible.

OUTPUT

Return ONLY JSON with ALL of these keys, in this exact order (fill unknowns with null or []):

{
  "top": null,
  "top_color": null,
  "top_features": [],
  "bottom": null,
  "bottom_color": null,
  "bottom_features": [],
  "dress": null,
  "dress_color": null,
  "dress_features": [],
  "outerwear": null,
  "outerwear_color": null,
  "outerwear_features": [],
  "shoes": null,
  "shoes_color": null,
  "accessories": [],
  "action": null,
  "style": null,
  "environment": null
}
"""
# edit the image path / prompts and just re-run this cell - no model reload needed
print(caption("data/sampled_images/image_00401.jpg", PROMPT))

In [ ]:
# SigLIP image embedder - store one vector per image next to its caption so the
# retrieval notebook can build the image index without recomputing. Same encoder
# the retriever uses. Runs on CPU so it stays out of the VLM's GPU.
import numpy as np
from transformers import SiglipModel

EMB_DIR = "data/siglip_image_emb"
os.makedirs(EMB_DIR, exist_ok=True)

_siglip = SiglipModel.from_pretrained("google/siglip2-base-patch16-224").to("cpu").eval()
_siglip_proc = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")

@torch.inference_mode()
def embed_image(image_path):
    image = Image.open(image_path).convert("RGB")
    inp = _siglip_proc(images=[image], return_tensors="pt").to("cpu")
    v = torch.nn.functional.normalize(_siglip.get_image_features(**inp), p=2, dim=-1)
    return v[0].cpu().numpy().astype("float32")

In [ ]:
import json
import os

OUT_DIR = "data/qwen3_vl_metadata"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(EMB_DIR, exist_ok=True)

for image_name in sorted(os.listdir("data/sampled_images")):
    image_path = os.path.join("data/sampled_images", image_name)
    stem = os.path.splitext(image_name)[0]
    out_path = os.path.join(OUT_DIR, f"{stem}.json")
    emb_path = os.path.join(EMB_DIR, f"{stem}.npy")
    if os.path.exists(out_path) and os.path.exists(emb_path):
        continue  # resumable: skip images already captioned AND embedded
    raw = caption(image_path, PROMPT)
    try:
        metadata = json.loads(raw)
    except json.JSONDecodeError:
        metadata = {"_raw": raw, "_parse_error": True}
    with open(out_path, "w") as f:
        json.dump(metadata, f, indent=2)
    np.save(emb_path, embed_image(image_path))

In [ ]:
for image_name in sorted(os.listdir("data/curated")):
    image_path = os.path.join("data/curated", image_name)
    stem = os.path.splitext(image_name)[0]
    out_path = os.path.join(OUT_DIR, f"{stem}.json")
    emb_path = os.path.join(EMB_DIR, f"{stem}.npy")
    if os.path.exists(out_path) and os.path.exists(emb_path):
        continue
    raw = caption(image_path, PROMPT)
    try:
        metadata = json.loads(raw)
    except json.JSONDecodeError:
        metadata = {"_raw": raw, "_parse_error": True}
    with open(out_path, "w") as f:
        json.dump(metadata, f, indent=2)
    np.save(emb_path, embed_image(image_path))